# Initialization

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

#Reading Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")
df.display()

#Data Transformations

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:

df = (
    df
    .withColumn("GEN",
    F.when(F.upper(F.col("GEN")).isin("M", "MALE"), "Male")
    .when(F.upper(F.col("GEN")).isin("F", "FEMALE"), "Female")
    .otherwise("n/a"))
)

## Changing Data Type and Birthday Validation

In [0]:
df = df.withColumn("BDATE",
        F.when(F.col("BDATE") > F.current_date(), None)
        .otherwise(F.to_date(F.col("BDATE"), "yyyy-MM-dd"))
    )

## Cleaning Customer ID

In [0]:
df = df.withColumn("cid",
                   F.when(col("cid").startswith("NAS"), 
                          F.substring(col("cid"), 4, F.length(col("cid")))
                          )
                   .otherwise(col("cid"))
                   )

## Renaming Columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.erp_customers")
)

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10